In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    precision_recall_curve
)

# ==================================================
# 1. Load dataset
# ==================================================

FILE_PATH = "session_dataset.csv"   # change if needed
TARGET = "any_addon_added"

df = pd.read_csv(FILE_PATH)

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

# ==================================================
# 2. Drop leakage + ID + memorization-like columns
# ==================================================

leakage_cols = [
    "actual_added_addon_names",
    "actual_added_addon_categories",
    "actual_added_addon_count",
    "actual_added_addon_value",
    "final_order_value",
    "cart_completion_score",
]

id_like_cols = [
    "session_id",
    "session_timestamp",
    "user_id",
    "restaurant_id",
    "restaurant_name",
]

# Raw text identity / near-lookup fields that can create brittle memorization
# Keep categories/count/value summaries, drop raw names/lists
high_cardinality_name_cols = [
    "base_cart_item_names",
    "recommended_addon_names",
]

drop_cols = [c for c in (leakage_cols + id_like_cols + high_cardinality_name_cols) if c in df.columns]

print("\nDropping columns:")
for c in drop_cols:
    print("-", c)

df = df.drop(columns=drop_cols)

print("\nShape after dropping columns:", df.shape)

# ==================================================
# 3. Basic target inspection
# ==================================================

print("\nTarget distribution:")
print(df[TARGET].value_counts(dropna=False))
print("Target rate:", round(df[TARGET].mean(), 4))

# ==================================================
# 4. Split features / target
# ==================================================

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

# Make sure target is numeric/binary
y = pd.to_numeric(y, errors="coerce").fillna(0).astype(int)

# ==================================================
# 5. Identify numeric and categorical columns
# ==================================================

numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("\nNumeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

# ==================================================
# 6. Train / test split
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ==================================================
# 7. Preprocessing
# ==================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# ==================================================
# 8. Model
# ==================================================
# class_weight helps with imbalance.
# Keep model simple but reasonable for a baseline.

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# ==================================================
# 9. Train
# ==================================================

pipeline.fit(X_train, y_train)

# ==================================================
# 10. Evaluate
# ==================================================

probs = pipeline.predict_proba(X_test)[:, 1]
preds_05 = (probs >= 0.5).astype(int)

auc = roc_auc_score(y_test, probs)
pr_auc = average_precision_score(y_test, probs)

print("\n=== Model performance ===")
print("ROC AUC:", round(auc, 4))
print("PR AUC :", round(pr_auc, 4))

print("\nClassification report at threshold = 0.50")
print(classification_report(y_test, preds_05, digits=3, zero_division=0))

# ==================================================
# 11. Top 20% targeting policy
# ==================================================

threshold_top20 = np.percentile(probs, 80)
top20_mask = probs >= threshold_top20

overall_conversion = y_test.mean()
targeted_conversion = y_test[top20_mask].mean()
absolute_lift = targeted_conversion - overall_conversion
relative_lift = targeted_conversion / overall_conversion if overall_conversion > 0 else np.nan

print("\n=== Targeting policy (top 20% highest-probability sessions) ===")
print("Threshold:", round(threshold_top20, 4))
print("Sessions selected:", int(top20_mask.sum()))
print("Overall conversion:", round(overall_conversion, 4))
print("Targeted conversion:", round(targeted_conversion, 4))
print("Absolute lift:", round(absolute_lift, 4))
print("Relative lift:", round(relative_lift, 4))

# ==================================================
# 12. Find a threshold with better precision/recall balance
# ==================================================
# Optional diagnostic: useful because threshold 0.50 may be poor in imbalanced tasks

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

# Avoid last precision/recall point mismatch
f1_scores = []
valid_thresholds = thresholds

for p, r in zip(precisions[:-1], recalls[:-1]):
    if p + r == 0:
        f1_scores.append(0)
    else:
        f1_scores.append(2 * p * r / (p + r))

f1_scores = np.array(f1_scores)
best_idx = int(np.argmax(f1_scores))
best_threshold = valid_thresholds[best_idx]
best_f1 = f1_scores[best_idx]

preds_best = (probs >= best_threshold).astype(int)

print("\n=== Best threshold by PR-curve F1 ===")
print("Best threshold:", round(float(best_threshold), 4))
print("Best F1:", round(float(best_f1), 4))
print(classification_report(y_test, preds_best, digits=3, zero_division=0))

# ==================================================
# 13. Feature importance extraction
# ==================================================

# Get feature names after preprocessing
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numeric_cols, cat_feature_names])

importances = pipeline.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\n=== Top 20 feature importances ===")
print(importance_df.head(20).to_string(index=False))

# ==================================================
# 14. Collapse one-hot features back to original columns
# ==================================================
# More useful for your benchmark answer: original variable-level importance

def base_feature_name(feat: str) -> str:
    # One-hot names look like: restaurant_cuisine_Italian
    # We map them back to the original column name if possible.
    for col in categorical_cols:
        prefix = col + "_"
        if feat.startswith(prefix):
            return col
    return feat

importance_df["base_feature"] = importance_df["feature"].apply(base_feature_name)

grouped_importance = (
    importance_df.groupby("base_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

print("\n=== Top 15 original variables ===")
print(grouped_importance.head(15).to_string(index=False))

# ==================================================
# 15. Save scoring outputs for ground-truth reference
# ==================================================

scored_test = X_test.copy()
scored_test[TARGET] = y_test.values
scored_test["predicted_probability"] = probs
scored_test["target_top20_aggressive_reco"] = top20_mask.astype(int)

# Save if needed
# scored_test.to_csv("q1_scored_test_sessions.csv", index=False)
# importance_df.to_csv("q1_feature_importance_expanded.csv", index=False)
# grouped_importance.to_csv("q1_feature_importance_grouped.csv", index=False)

# ==================================================
# 16. Print a clean summary for answer file
# ==================================================

top_vars = grouped_importance.head(10)["base_feature"].tolist()

print("\n=== Reference summary for answer file ===")
print({
    "overall_conversion": round(float(overall_conversion), 4),
    "roc_auc": round(float(auc), 4),
    "pr_auc": round(float(pr_auc), 4),
    "top20_threshold": round(float(threshold_top20), 4),
    "top20_conversion": round(float(targeted_conversion), 4),
    "absolute_lift": round(float(absolute_lift), 4),
    "relative_lift": round(float(relative_lift), 4),
    "top_variables": top_vars
})

Original shape: (100000, 57)

Columns:
['session_id', 'session_timestamp', 'user_id', 'restaurant_id', 'restaurant_name', 'user_segment', 'user_city', 'user_preferred_cuisine', 'user_veg_preference', 'user_price_sensitivity', 'user_order_frequency_30d', 'user_avg_order_value', 'user_recency_days', 'num_past_orders_at_restaurant', 'user_addon_acceptance_rate', 'user_preferred_addon_category', 'restaurant_city', 'restaurant_cuisine', 'restaurant_type', 'restaurant_online_order', 'restaurant_price_tier', 'restaurant_rating', 'restaurant_is_chain', 'restaurant_delivery_time_avg', 'restaurant_avg_orders_per_day', 'hour', 'day_of_week', 'meal_time', 'is_weekend', 'has_offer', 'weather_condition', 'traffic_density', 'is_festival_day', 'estimated_delivery_time', 'delivery_zone', 'session_engagement_score', 'base_cart_item_names', 'base_cart_item_categories', 'base_cart_item_count', 'base_cart_value', 'cart_has_drink', 'cart_has_dessert', 'cart_has_side', 'cart_completion_score', 'recommended_a